In [1]:
!pip uninstall -y datasets huggingface_hub
!pip install datasets huggingface_hub
import random
import numpy as np
import torch


from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score



Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
Found existing installation: huggingface_hub 1.18.0
Uninstalling huggingface_hub-1.18.0:
  Successfully uninstalled huggingface_hub-1.18.0
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.18.0-py3-none-any.whl.metadata (14 kB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
Using cached huggingface_hub-1.18.0-py3-none-any.whl (684 kB)


In [2]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)

In [3]:
!pip install -U datasets huggingface_hub

In [4]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

4.8.5
1.18.0


In [5]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [6]:
sample=dataset["train"][0]
print("Label:",sample["label"])
print("0 is negative and 1 is positive")
print("\nReview:",sample["text"][:500],"...")

Label: 0
0 is negative and 1 is positive

Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent ...


In [7]:
train_size=2000
test_size=500

small_train=dataset["train"].shuffle(seed=42).select(range(train_size))
small_test=dataset["test"].shuffle(seed=42).select(range(test_size))

print(small_train)
print(small_test)

Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 500
})


In [8]:
split=small_train.train_test_split(test_size=0.2,seed=42)
train=split["train"]
val=split["test"]

print(train)
print(len(val))

Dataset({
    features: ['text', 'label'],
    num_rows: 1600
})
400


In [9]:
MODEL_NAME='bert-base-uncased'
MAX_LENGTH=256

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
def tokenize(batch):
  return tokenizer(batch["text"],padding=True,truncation=True,max_length=MAX_LENGTH)

train_tokenized=train.map(tokenize,batched=True,batch_size=None)
val_tokenized=val.map(tokenize,batched=True,batch_size=None)
test_tokenized=small_test.map(tokenize,batched=True,batch_size=None)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [11]:
model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
def compute_metrics(eval_pred):
  predictions,labels=eval_pred
  predictions=np.argmax(predictions,axis=1)
  return{
  "f1": f1_score(labels,predictions),
  "accuracy":accuracy_score(labels,predictions),
  "precision":precision_score(labels,predictions),
  "recall":recall_score(labels,predictions)
  }

In [13]:
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
training_args=TrainingArguments(
    output_dir="./bert_imdb_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
    logging_steps=50
)

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy,Precision,Recall
1,0.374549,0.296945,0.863510,0.877500,0.906433,0.824468
2,0.221372,0.257041,0.907652,0.912500,0.900524,0.914894


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.3567780065536499, metrics={'train_runtime': 66.0948, 'train_samples_per_second': 48.415, 'train_steps_per_second': 3.026, 'total_flos': 420977688576000.0, 'train_loss': 0.3567780065536499, 'epoch': 2.0})

In [16]:
final_results=trainer.evaluate(test_tokenized)
print(final_results)

{'eval_loss': 0.2785387337207794, 'eval_f1': 0.8812877263581489, 'eval_accuracy': 0.882, 'eval_precision': 0.8725099601593626, 'eval_recall': 0.8902439024390244, 'eval_runtime': 1.9186, 'eval_samples_per_second': 260.612, 'eval_steps_per_second': 16.679, 'epoch': 2.0}


In [17]:
id2label={"0":"Negative","1":"Positive"}

def predict_sentiment(text):
  inputs=tokenizer(text,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors="pt").to(model.device)
  model.eval()
  with torch.no_grad():
    logits=model(**inputs).logits
  predicted_class_id=logits.argmax().item()
  return {"label":id2label[str(predicted_class_id)],
          "confidence":float(logits.softmax(dim=-1).max())
          }

reviews=["This movie is awesome",
         "It was boring and too long.",
         "It was okay-watchable yk"]

for review in reviews:
  print(predict_sentiment(review))

{'label': 'Positive', 'confidence': 0.9091436266899109}
{'label': 'Negative', 'confidence': 0.8726732134819031}
{'label': 'Positive', 'confidence': 0.5724837779998779}


In [18]:
SAVE_DIR="./imdb_bert"
trainer.save_model(SAVE_DIR)

tokenizer.save_pretrained(SAVE_DIR)
SAVE_DIR

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'./imdb_bert'

In [29]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification
SAVE_DIR="./imdb_bert"

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Model and tokenizer reloaded on:",device)

id2label = {0: "Negative", 1: "Positive"}
MAX_LENGTH=256
def predict_sentiment(text):
    inputs = tokenizer(text, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_class_id = logits.argmax().item()
    probs = logits.softmax(dim=-1).tolist()[0]
    return {"label": id2label[predicted_class_id], "confidence": float(logits.softmax(dim=-1).max())}

def classify_review(review):
    if not review.strip():
        return "Please enter a review."

    result = predict_sentiment(review)

    label = result["label"]
    confidence = result["confidence"]

    emoji = "😊" if label == "Positive" else "😞"

    return f"{emoji} {label} ({confidence*100:.2f}%)"

demo = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(lines=5, label="Review",placeholder="Type a movie review here"),
    outputs=gr.Markdown(label="Prediction"),
    title="🎬IMDB Sentiment Classifier",
    description="Fine-tuned BERT predicting whether a movie review is positive or negative",
    flagging_mode="never",
    examples=["This movie is awesome",
         "It was boring and too long.",
         "It was okay-some good moments but overall forgettable"]
)

demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model and tokenizer reloaded on: cuda
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd9b9f3bf738860ebe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
